# set up

In [ ]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"chujikgch","key":"b8386abbd0a8d7e4a9f9cdc8d9047a2d"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c super-ai-engineer-season-6-ocr-2569

100% 474M/474M [00:04<00:00, 119MB/s]



In [ ]:
!unzip super-ai-engineer-season-6-ocr-2569.zip

Archive:  super-ai-engineer-season-6-ocr-2569.zip
  inflating: final_data/images/constituency_10_1.png  
  inflating: final_data/images/constituency_10_10.png  
  inflating: final_data/images/constituency_10_10_page2.png  
  inflating: final_data/images/constituency_10_11.png  
  inflating: final_data/images/constituency_10_11_page2.png  
  inflating: final_data/images/constituency_10_11_page3.png  
  inflating: final_data/images/constituency_10_12.png  
  inflating: final_data/images/constituency_10_12_page2.png  
  inflating: final_data/images/constituency_10_12_page3.png  
  inflating: final_data/images/constituency_10_13.png  
  inflating: final_data/images/constituency_10_13_page2.png  
  inflating: final_data/images/constituency_10_14.png  
  inflating: final_data/images/constituency_10_14_page2.png  
  inflating: final_data/images/constituency_10_14_page3.png  
  inflating: final_data/images/constituency_10_16.png  
  inflating: final_data/images/constituency_10_16_page2.png  
 

In [ ]:
import requests

print("🔍 กำลังเจาะระบบขอดูรายชื่อโมเดลด้วย HTTP Request โดยตรง...")

api_key = "sk-CMBx2RAkWCxGJb3Sxmyzz00DYS38uiehaAS15EAcwLHrai6T"
url = "https://api.opentyphoon.ai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# 1. ยิง Request สายตรงไปที่เซิร์ฟเวอร์
response = requests.get(url, headers=headers)

# 2. เช็คว่าเซิร์ฟเวอร์ตอบรับโอเคไหม (Status 200 = OK)
if response.status_code == 200:
    data = response.json()

    # ดักจับข้อมูลเผื่อเซิร์ฟเวอร์ส่งมาเป็น List หรือ Dict
    models = data if isinstance(data, list) else data.get("data", [])

    print("\n🎯 รายชื่อโมเดลที่มีคำว่า 'vision':")
    found_vision = False

    for m in models:
        # สกัดชื่อโมเดลออกมา
        model_name = m.get("id", "Unknown") if isinstance(m, dict) else str(m)

        # กรองหาคำว่า vision
        if "" in model_name.lower():
            print(f"✅ {model_name}")
            found_vision = True

    if not found_vision:
        print("⚠️ ไม่เจอโมเดลที่มีคำว่า vision เลย ลองเช็คหน้าเว็บแพลตฟอร์มดูอีกทีครับ")

else:
    print(f"❌ ดึงข้อมูลไม่สำเร็จ: รหัส {response.status_code}")
    print(f"รายละเอียด: {response.text}")

🔍 กำลังเจาะระบบขอดูรายชื่อโมเดลด้วย HTTP Request โดยตรง...

🎯 รายชื่อโมเดลที่มีคำว่า 'vision':
⚠️ ไม่เจอโมเดลที่มีคำว่า vision เลย ลองเช็คหน้าเว็บแพลตฟอร์มดูอีกทีครับ


In [ ]:
!pip install -q openai tqdm pandas

In [ ]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import requests
import json
import re
import glob
import os
import time
from rapidfuzz import process, fuzz
from tqdm.auto import tqdm

print("⏳ กำลังเตรียมระบบ Ultimate Typhoon OCR + Hybrid Logic...")

# ==========================================
# 🔑 1. ตั้งค่า API (อย่าลืมใช้ Key เส้นใหม่นะครับ!)
# ==========================================
API_KEY = "sk-CMBx2RAkWCxGJb3Sxmyzz00DYS38uiehaAS15EAcwLHrai6T"
MODEL_NAME = "typhoon-ocr"

# ==========================================
# 🧩 2. ฟังก์ชันหลัก (จาก Document ของ Typhoon)
# ==========================================
def extract_text_with_typhoon(image_path):
    url = "https://api.opentyphoon.ai/v1/ocr"
    try:
        with open(image_path, 'rb') as file:
            files = {'file': file}
            data = {
                'model': MODEL_NAME,
                'task_type': 'default',
                'max_tokens': '2000',
                'temperature': '0.1', # ลดความหลอน
                'top_p': '0.6',
                'repetition_penalty': '1.2'
            }
            headers = {'Authorization': f'Bearer {API_KEY}'}

            response = requests.post(url, files=files, data=data, headers=headers)

            if response.status_code == 200:
                result = response.json()
                extracted_texts = []
                for page_result in result.get('results', []):
                    if page_result.get('success') and page_result.get('message'):
                        content = page_result['message']['choices'][0]['message']['content']
                        try:
                            parsed = json.loads(content)
                            text = parsed.get('natural_text', content)
                        except json.JSONDecodeError:
                            text = content
                        extracted_texts.append(text)
                return '\n'.join(extracted_texts)
            else:
                return None
    except Exception as e:
        print(f"Error reading {image_path}: {e}")
        return None

# ==========================================
# 🧠 3. สมองกล: สกัดตัวเลขและซ่อมคำผิด
# ==========================================
def extract_last_number(text):
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)
    t_clean = re.sub(r'\s*,\s*', '', t)
    nums = re.findall(r'\d+', t_clean)
    if nums: return int(nums[-1])
    return 0

def fix_thai_typos(text):
    t = re.sub(r'[\.\-\*\_]', '', text).strip()
    if "เพื่อ" not in t: t = t.replace("เพือ", "เพื่อ")
    if "ภูมิใจ" not in t: t = t.replace("ภูมใจ", "ภูมิใจ")
    if "ประชา" not in t: t = t.replace("ประขา", "ประชา")
    if "ชาติ" not in t: t = t.replace("ชาต ", "ชาติ ")
    if "อนาคต" not in t: t = t.replace("อนาคด", "อนาคต")
    if "ใหม่" not in t: t = t.replace("ใหม", "ใหม่")
    if "พร้อม" not in t: t = t.replace("พรอม", "พร้อม")
    if "กรีน" not in t: t = t.replace("กรน", "กรีน").replace("กร์น", "กรีน")
    if "ไทย" not in t: t = t.replace("ทย", "ไทย")
    return t

# ==========================================
# 🚀 4. ระบบจัดการหลักกวาด 847 ไฟล์
# ==========================================
df_template = pd.read_csv('/content/data/submission_template.csv') # ใช้ V.2 นะครับ!
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

file_list = sorted(glob.glob("/content/data/images/*.png"))
final_submission = {row_id: 0 for row_id in df_template['id']}

print(f"📦 เริ่มกวาดตารางด้วย Typhoon OCR ทั้งหมด {len(file_list)} ไฟล์...")

for img_path in tqdm(file_list):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    # ยิง API เพื่อดึงข้อความ (Auto-retry 3 ครั้งถ้าเน็ตหลุด)
    raw_text = None
    for attempt in range(3):
        raw_text = extract_text_with_typhoon(img_path)
        if raw_text: break
        time.sleep(2)

    if not raw_text: continue # ข้ามถ้ายิงไม่ผ่านจริงๆ

    # ให้สมองกลเราจับคู่จาก Text ที่ Typhoon อ่านมาได้
    votes_dict = {}
    for line in raw_text.split('\n'):
        if not line.strip(): continue

        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        text_fixed = fix_thai_typos(text_no_num)

        best_match = process.extractOne(text_fixed, valid_parties, scorer=fuzz.token_set_ratio)

        # ตัดสินใจจับคู่ (ลดความเข้มลงเหลือ 45 เพราะ Typhoon ช่วยอ่านให้แม่นระดับนึงแล้ว)
        if best_match and best_match[1] > 45:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    # หยอดคะแนนลงตาราง Template
    for party, vote in votes_dict.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

# เซฟผลลัพธ์
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_typhoon_ocr.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบ Typhoon OCR สมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_typhoon_ocr.csv")
print("="*40)

⏳ กำลังเตรียมระบบ Ultimate Typhoon OCR + Hybrid Logic...
📦 เริ่มกวาดตารางด้วย Typhoon OCR ทั้งหมด 847 ไฟล์...


  0%|          | 0/847 [00:00<?, ?it/s]


✅ ปิดจ๊อบ Typhoon OCR สมบูรณ์! เติมคะแนนได้ 6/10053 แถว
📂 เซฟไฟล์พร้อมส่ง: /content/submission_typhoon_ocr.csv


จบ typhoonใช้ไม่ได้

In [ ]:

!pip install qwen_vl_utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 74.8 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
from PIL import Image
import torch
import re
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info
import glob
import os
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict


In [ ]:
# ==========================================
# โค้ดสำหรับโหลด Typhoon OCR ฉบับสมบูรณ์
# ==========================================
model_id = "typhoon-ai/typhoon-ocr1.5-2b"

print("⏳ กำลังโหลดโมเดลภาษาไทยเฉพาะทาง (Typhoon OCR)...")

# 1. โหลด Processor

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

# 2. โหลด Model ด้วยคลาสใหม่ล่าสุดของยุค 5.x
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,   # ← เปลี่ยนจาก bfloat16
    device_map="auto",
    trust_remote_code=True
)
model = torch.compile(model)     # ← Torch compile เร็วขึ้น ~20-30%

print("✅ โหลดเสร็จสิ้น! AutoModelForImageTextToText ทำงานสมบูรณ์แบบ!")

⏳ กำลังโหลดโมเดลภาษาไทยเฉพาะทาง (Typhoon OCR)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

✅ โหลดเสร็จสิ้น! AutoModelForImageTextToText ทำงานสมบูรณ์แบบ!


In [ ]:
def ocr_page_typhoon(img_path):
    img = Image.open(img_path).convert('RGB')
    img.thumbnail((672, 672), Image.Resampling.LANCZOS)

    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Extract all text from the image."},
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=imgs, padding=False, return_tensors="pt").to("cuda")
    if "mm_token_type_ids" in inputs:
        inputs.pop("mm_token_type_ids")

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1500, do_sample=False,
                             repetition_penalty=1.1, use_cache=True)
    return processor.batch_decode(
        [out[0][inputs.input_ids.shape[-1]:]], skip_special_tokens=True
    )[0]

In [ ]:
import re

THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

def thai_to_arabic(text):
    if not isinstance(text, str): return str(text)
    return text.translate(THAI_DIGITS)

def extract_votes_for_parties(ocr_text, party_names):
    ocr_text = thai_to_arabic(ocr_text)
    lines = [l.strip() for l in ocr_text.split("\n") if l.strip()]
    party_names = [p for p in party_names if isinstance(p, str)]
    results = {p: 0 for p in party_names}

    for i, line in enumerate(lines):
        for party in party_names:
            if party in line:
                window = " ".join(lines[i:i+3])
                window = thai_to_arabic(window)
                nums = re.findall(r"\d[\d,]*", window)
                nums = [int(n.replace(",","")) for n in nums if n.replace(",","").isdigit()]
                valid = [n for n in nums if 1 <= n <= 200000]
                if valid:
                    results[party] = max(valid)
                    break
    return results

In [ ]:
df_template = pd.read_csv("/content/final_data/submission_template_v3.csv")

# ==========================================
# 🛠️ แผนกซ่อมแซม: สร้างคอลัมน์ที่หายไปขึ้นมาใหม่
# ==========================================
if 'doc_id' not in df_template.columns:
    # หั่นข้อความที่เครื่องหมาย _ ตัวสุดท้าย แล้วแยกเป็น 2 คอลัมน์
    split_cols = df_template['id'].str.rsplit('_', n=1, expand=True)
    df_template['doc_id'] = split_cols[0]
    df_template['party_name'] = split_cols[1]
# ==========================================

# ตอนนี้มีคอลัมน์ doc_id แล้ว groupby จะไม่พังอีกต่อไป!
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()

# แก้ไข path ให้มี / ก่อน *.png ด้วยนะครับ ไม่งั้นมันจะหาไฟล์ไม่เจอ
file_list = sorted(glob.glob("/content/final_data/images/*.png"))
print(f"🚀 ภาพทั้งหมด {len(file_list)} ไฟล์, docs {len(doc_to_parties)} docs")

def image_to_doc_id(img_path):
    name = os.path.basename(img_path).replace(".png","")
    return re.sub(r"_page\d+$", "", name)

doc_to_files = defaultdict(list)
for f in file_list:
    doc_to_files[image_to_doc_id(f)].append(f)

votes_dict = {}

for doc_id, parties in tqdm(doc_to_parties.items()):
    pages = sorted(doc_to_files.get(doc_id, []))
    if not pages:
        continue

    full_text = ""
    for page_path in pages:
        try:
            full_text += ocr_page_typhoon(page_path) + "\n"
        except Exception as e:
            print(f"❌ {page_path}: {e}")

    # โค้ดตรงนี้จะทำงานได้ปกติ เพราะเราสร้าง doc_id กับ party_name ไว้แล้ว
    id_map = df_template[df_template["doc_id"]==doc_id].set_index("party_name")["id"].to_dict()
    votes_map = extract_votes_for_parties(full_text, parties)

    for party, votes in votes_map.items():
        row_id = id_map.get(party)
        if row_id:
            votes_dict[row_id] = votes

    # Checkpoint ทุก 20 docs (แต่ในโค้ดเขียน % 100 ไว้ แนะนำเปลี่ยนเป็น % 20 ครับจะได้เซฟบ่อยขึ้น)
    if len(votes_dict) % 100 == 0 and votes_dict:
        df_ck = df_template[["id"]].copy()
        df_ck["votes"] = df_ck["id"].map(votes_dict).fillna(0).astype(int)
        df_ck.to_csv("/content/submission_checkpoint.csv", index=False)
        print(f"💾 Checkpoint: {df_ck['votes'].gt(0).sum()} rows filled")

# submission สุดท้าย
df_out = df_template[["id"]].copy()
df_out["votes"] = df_out["id"].map(votes_dict).fillna(0).astype(int)
df_out.to_csv("/content/submission.csv", index=False)
filled = df_out["votes"].gt(0).sum()
print(f"\n✅ Done! {filled}/{len(df_out)} rows ({filled/len(df_out)*100:.1f}%)")
print(df_out[df_out["votes"]>0].head(10))

🚀 ภาพทั้งหมด 846 ไฟล์, docs 300 docs


  0%|          | 0/300 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
import pandas as pd
import re

# โหลด template
df_template = pd.read_csv('/content/data/submission_template.csv')

# โหลด all_results (รองรับกรณี session หลุด)
try:
    df_results = pd.DataFrame(all_results)
except:
    df_results = pd.read_csv('/content/all_results_final.csv')

print(f"Template rows: {len(df_template)}")
print(f"OCR results rows: {len(df_results)}")

# image_id → doc_id เช่น "constituency_10_1_page2.png" → "constituency_10_1"
def image_to_doc_id(image_id):
    name = image_id.replace('.png', '')
    return re.sub(r'_page\d+$', '', name)

df_results['doc_id'] = df_results['image_id'].apply(image_to_doc_id)

# สร้าง lookup (doc_id, party) → votes
votes_lookup = {}
for _, row in df_results.iterrows():
    key = (row['doc_id'], row['party'])
    if key not in votes_lookup or votes_lookup[key] == 0:
        votes_lookup[key] = row['votes']

# Match กับ template
df_template['votes'] = df_template.apply(
    lambda row: votes_lookup.get((row['doc_id'], row['party_name']), 0), axis=1
)

filled = df_template['votes'].gt(0).sum()
print(f"Filled: {filled}/{len(df_template)} ({filled/len(df_template)*100:.1f}%)")

# ✅ ส่งแค่ id กับ votes เท่านั้น
df_template[['id', 'votes']].to_csv('/content/submission.csv', index=False)
print("✅ Saved: /content/submission.csv")
print(df_template[['id', 'votes']].head(10))

Template rows: 10053
OCR results rows: 731
Filled: 78/10053 (0.8%)
✅ Saved: /content/submission.csv
                     id  votes
0   constituency_10_1_1      0
1   constituency_10_1_2      0
2   constituency_10_1_3      0
3   constituency_10_1_4      0
4   constituency_10_1_5      0
5   constituency_10_1_6      0
6   constituency_10_1_7      0
7   constituency_10_1_8      0
8   constituency_10_1_9      0
9  constituency_10_1_10      0


pyte

In [ ]:
!pip install easyocr pytesseract rapidfuzz opencv-python pillow pandas tqdm
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-tha

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 37.6 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tesseract-ocr-tha
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 899 kB of archives.
After this operation, 1,087 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/univ

In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import easyocr
import pytesseract
from rapidfuzz import fuzz
from collections import Counter

In [ ]:
IMAGE_DIR = "/content/data/images"
SUBMISSION_PATH = "/content/data/submission_template.csv"
OUTPUT_PATH = "/content/submission.csv"

reader = easyocr.Reader(['th','en'], gpu=True)

In [ ]:
def clean_number(text):
    if not text:
        return ""

    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_map)

    text = text.replace(',', '')

    text = text.replace('O','0').replace('o','0')
    text = text.replace('I','1').replace('l','1')
    text = text.replace('B','8')

    text = re.sub(r'[^0-9]', '', text)

    return text

In [ ]:
def group_images(image_dir):
    groups = {}

    for f in os.listdir(image_dir):
        if not f.endswith(".png"):
            continue

        base = re.sub(r'_page\d+', '', f.replace('.png',''))

        groups.setdefault(base, []).append(os.path.join(image_dir, f))

    return groups

In [ ]:
def crop_vote_column(img):
    h, w = img.shape[:2]
    return img[:, int(w*0.65):]

In [ ]:
def tesseract_worker(path):
    img = cv2.imread(path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    crop = crop_vote_column(thresh)

    text = pytesseract.image_to_string(crop, lang='tha+eng', config="--psm 6")
    lines = text.split("\n")

    nums = []
    for i, line in enumerate(lines):
        num = clean_number(line)
        if num:
            nums.append((i*50, num))

    return path, nums

In [ ]:
def easyocr_process(img):
    result = reader.readtext(img, detail=1)

    rows = []
    for bbox, text, conf in result:
        y = bbox[0][1]
        num = clean_number(text)

        if num:
            rows.append((y, num))

    return sorted(rows, key=lambda x: x[0])

In [ ]:
def merge_votes(easy_rows, tess_rows):
    merged = []

    max_len = max(len(easy_rows), len(tess_rows))

    for i in range(max_len):
        candidates = []

        if i < len(easy_rows):
            candidates.append(easy_rows[i][1])

        if i < len(tess_rows):
            candidates.append(tess_rows[i][1])

        if candidates:
            vote = Counter(candidates).most_common(1)[0][0]
        else:
            vote = ""

        merged.append(vote)

    return merged

In [ ]:
groups = group_images(IMAGE_DIR)
tasks = [(doc_id, paths) for doc_id, paths in groups.items()]

# 🔥 เตรียม tesseract parallel
all_paths = []
for _, paths in tasks:
    all_paths.extend(paths)

num_workers = min(4, cpu_count())

with Pool(num_workers) as p:
    tess_results = dict(tqdm(p.imap(tesseract_worker, all_paths), total=len(all_paths)))

# 🔥 EasyOCR (GPU main)
doc_results = []

for doc_id, paths in tqdm(tasks):
    doc_votes = []

    for path in sorted(paths):
        img = cv2.imread(path)
        crop = crop_vote_column(img)

        easy_rows = easyocr_process(crop)
        tess_rows = tess_results.get(path, [])

        merged = merge_votes(easy_rows, tess_rows)

        doc_votes.extend(merged)

    doc_results.append((doc_id, doc_votes))

100%|██████████| 300/300 [10:23<00:00,  2.08s/it]


In [ ]:
submission = pd.read_csv(SUBMISSION_PATH)

all_votes = []

doc_results = sorted(doc_results, key=lambda x: x[0])

for _, votes in doc_results:
    for v in votes:
        if not v:
            v = "0"

        if len(v) > 6:
            v = v[:6]

        all_votes.append(v)

submission["votes"] = all_votes[:len(submission)]
submission.to_csv("submission.csv", index=False)

print("✅ DONE")

✅ DONE


Y-Axis Grouping & Fuzzy Match

In [ ]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 33.5 MB/s eta 0:00:00


In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import easyocr
from rapidfuzz import process, fuzz
import re
import glob
import os
from tqdm.auto import tqdm

In [ ]:


print("⏳ กำลังโหลด EasyOCR...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

# 1. โหลด Template และสร้าง Lookup Tables
df_template = pd.read_csv('/content/data/submission_template.csv')
# Map 1: doc_id -> รายชื่อพรรคที่เป็นไปได้ในหน้านั้น
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
# Map 2: (doc_id, party_name) -> id แถวของ Submission
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    """ฟังก์ชันดึงตัวเลข (V.3 ล้างบาง ด.เด็ก ปลอมตัว!)"""
    # 1. ปราบ ด.เด็ก: ถ้า 'ด' อยู่ติดกับตัวเลข(ไทย/อารบิก) หรือลูกน้ำ ให้เปลี่ยนเป็น '1'
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', text) # เช็คข้างหน้า
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)    # เช็คข้างหลัง

    # 2. ปราบ โอ/อ่าง: เผื่อ AI อ่านเลข '0' เป็นตัว 'O' หรือ 'อ'
    t = re.sub(r'(?<=[0-9๐-๙,])[oOอ]', '0', t)
    t = re.sub(r'[oOอ](?=[0-9๐-๙,])', '0', t)

    # 3. แปลงเลขไทยเป็นอารบิก
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)

    # 4. ดูดลูกน้ำให้ติดกัน ("14,011" จะไม่แหว่ง)
    t_clean = re.sub(r'\s*,\s*', '', t)

    # 5. สกัดเลขก้อนขวาสุด
    nums = re.findall(r'\d+', t_clean)
    if nums:
        return int(nums[-1])
    return 0



⏳ กำลังโหลด EasyOCR...


In [ ]:
def process_image_adaptive(img_path, valid_parties, doc_type):
    """อ่านภาพ จัดกลุ่มแนวนอนแบบแยกคอลัมน์ซ้าย-ขวา!"""
    result = reader.readtext(
        img_path,
        mag_ratio=2.0,
        contrast_ths=0.2,
        adjust_contrast=0.6
    )

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []

    # 🎯 ท่าไม้ตายใหม่: หั่นคอลัมน์ซ้าย-ขวา (เฉพาะ Party List)
    if doc_type == 'party_list':
        # แบ่งเป็น 2 กอง ซ้าย และ ขวา โดยใช้พิกัด X กลางหน้ากระดาษ (ประมาณ X=1100)
        left_col = [item for item in lines if item['x'] < 1100]
        right_col = [item for item in lines if item['x'] >= 1100]

        # รันการมัดรวมบรรทัด (แกน Y) แยกกันทีละคอลัมน์!
        for col_items in [left_col, right_col]:
            current_group = []
            for item in col_items:
                if not current_group:
                    current_group.append(item)
                else:
                    if abs(item['y'] - current_group[-1]['y']) < 15: # ตาถี่สำหรับ Party List
                        current_group.append(item)
                    else:
                        current_group.sort(key=lambda i: i['x'])
                        grouped_lines.append(" ".join([i['text'] for i in current_group]))
                        current_group = [item]
            if current_group:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))

    else:
        # ส.ส.เขต มีคอลัมน์เดียว ทำเหมือนเดิม ใช้ตาข่าย 30px
        current_group = []
        for item in lines:
            if not current_group:
                current_group.append(item)
            else:
                if abs(item['y'] - current_group[-1]['y']) < 30:
                    current_group.append(item)
                else:
                    current_group.sort(key=lambda i: i['x'])
                    grouped_lines.append(" ".join([i['text'] for i in current_group]))
                    current_group = [item]
        if current_group:
            current_group.sort(key=lambda i: i['x'])
            grouped_lines.append(" ".join([i['text'] for i in current_group]))

    # --- จับคู่ชื่อพรรคและดึงคะแนน (เหมือนเดิม) ---
    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        best_match = process.extractOne(text_no_num, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 50:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

In [ ]:
# ==========================================
# 🚀 ลุยรันทั้งโฟลเดอร์แบบปลอดภัย 100%
# ==========================================
file_list = sorted(glob.glob("/content/data/images/*.png"))
final_submission = {row_id: 0 for row_id in df_template['id']}

print(f"🚀 เริ่มสกัดและจับคู่ข้อมูล {len(file_list)} ไฟล์...")

for img_path in tqdm(file_list):
    filename = os.path.basename(img_path)
    # ตัด _page ออกเพื่อหา doc_id หลัก
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    # เรียกฟังก์ชัน AI
    extracted_votes = process_image_with_fuzz(img_path, valid_parties)

    # หยอดคะแนนลง Submission ID ที่ถูกต้องแบบเป๊ะๆ!
    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            # ถ้ามีหลายหน้าแล้วเจอพรรคเดิมซ้ำ เอาคะแนนที่เยอะกว่า (เผื่อหน้าแรกอ่านพลาด)
            final_submission[row_id] = max(final_submission[row_id], vote)

# สร้าง DataFrame และเซฟผลลัพธ์
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_fuzz.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print(f"\n✅ เสร็จสมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_fuzz.csv")

🚀 เริ่มสกัดและจับคู่ข้อมูล 847 ไฟล์...


  0%|          | 0/847 [00:00<?, ?it/s]


✅ เสร็จสมบูรณ์! เติมคะแนนได้ 8908/10053 แถว
📂 เซฟไฟล์พร้อมส่ง: /content/submission_fuzz.csv


In [ ]:
# สร้าง DataFrame และเซฟผลลัพธ์
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
# เซฟผลลัพธ์เป็นไฟล์ชื่อใหม่ จะได้ไม่สับสนครับ!
df_out.to_csv("/content/submission_adaptive.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print(f"\n✅ เสร็จสมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_fuzz.csv")


✅ เสร็จสมบูรณ์! เติมคะแนนได้ 8908/10053 แถว
📂 เซฟไฟล์พร้อมส่ง: /content/submission_fuzz.csv


# score 0.5247

score สูงงสุดตอนนี้

In [ ]:
import pandas as pd
import easyocr
from rapidfuzz import process, fuzz
import re
import glob
import os
from tqdm.auto import tqdm

print("⏳ กำลังโหลด EasyOCR V.Final (Hybrid Pipeline)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

df_template = pd.read_csv('/content/data/submission_template.csv')
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    """ฟังก์ชันดึงตัวเลข (V.Final ปราบ ด.เด็ก + อังกฤษปลอมตัว)"""
    # 1. ปราบภาษาอังกฤษที่หน้าตาเหมือนตัวเลข
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')

    # 2. ปราบ ด.เด็ก ปลอมตัว
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)

    # 3. แปลงเลขไทย -> อารบิก
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)

    # 4. ดูดลูกน้ำให้ติดกัน
    t_clean = re.sub(r'\s*,\s*', '', t)

    # 5. ดึงก้อนตัวเลข
    nums = re.findall(r'\d+', t_clean)
    if nums:
        return int(nums[-1])
    return 0

def process_image_hybrid(img_path, valid_parties, doc_type):
    result = reader.readtext(
        img_path,
        mag_ratio=2.0,
        contrast_ths=0.2,
        adjust_contrast=0.6
    )

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []
    current_group = []

    # 🎯 ใช้ตาข่ายแบบตัวเลข (ดีที่สุดจากที่ทดสอบมา)
    # ขยายให้กว้างขึ้นเพื่อสู้กับกระดาษเอียง
    y_threshold = 25 if doc_type == 'party_list' else 35

    for item in lines:
        if not current_group:
            current_group.append(item)
        else:
            if abs(item['y'] - current_group[-1]['y']) < y_threshold:
                current_group.append(item)
            else:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))
                current_group = [item]

    if current_group:
        current_group.sort(key=lambda i: i['x'])
        grouped_lines.append(" ".join([i['text'] for i in current_group]))

    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        best_match = process.extractOne(text_no_num, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 50:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

# ==========================================
# 🚀 ลุยรันทั้งโฟลเดอร์แบบ Hybrid
# ==========================================
file_list = sorted(glob.glob("/content/data/images/*.png"))
final_submission = {row_id: 0 for row_id in df_template['id']}

print(f"🚀 เริ่มสแกนด้วยระบบ Hybrid {len(file_list)} ไฟล์...")

for img_path in tqdm(file_list):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    doc_type = "party_list" if "party_list" in filename else "constituency"
    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    extracted_votes = process_image_hybrid(img_path, valid_parties, doc_type)

    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_hybrid.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบ Hybrid สมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_hybrid.csv")
print("="*40)

⏳ กำลังโหลด EasyOCR V.Final (Hybrid Pipeline)...
🚀 เริ่มสแกนด้วยระบบ Hybrid 847 ไฟล์...


  0%|          | 0/847 [00:00<?, ?it/s]


✅ ปิดจ๊อบ Hybrid สมบูรณ์! เติมคะแนนได้ 8883/10053 แถว
📂 เซฟไฟล์พร้อมส่ง: /content/submission_hybrid.csv


# score 0.5026

In [ ]:
import pandas as pd
import easyocr
from rapidfuzz import process, fuzz
import re
import glob
import os
from tqdm.auto import tqdm

print("⏳ กำลังโหลด EasyOCR Hybrid (V.3 + อัปเดต Template V.2)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

# 🎯 จุดเปลี่ยนสำคัญ: เปลี่ยนมาใช้ Template V.2 ที่ผู้จัดเพิ่งแจกให้!
# (อย่าลืมแก้ Path ให้ตรงกับที่คุณ Chuji อัปโหลดไฟล์ไว้นะครับ)
df_template = pd.read_csv('/content/final_data/submission_template_v3.csv')
if 'doc_id' not in df_template.columns:
    split_cols = df_template['id'].str.rsplit('_', n=1, expand=True)
    df_template['doc_id'] = split_cols[0]
    df_template['party_name'] = split_cols[1]

doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)
    t_clean = re.sub(r'\s*,\s*', '', t)
    nums = re.findall(r'\d+', t_clean)
    if nums: return int(nums[-1])
    return 0

def fix_thai_typos(text):
    t = re.sub(r'[\.\-\*\_]', '', text).strip()
    if "เพื่อ" not in t: t = t.replace("เพือ", "เพื่อ")
    if "ภูมิใจ" not in t: t = t.replace("ภูมใจ", "ภูมิใจ")
    if "ประชา" not in t: t = t.replace("ประขา", "ประชา")
    if "ชาติ" not in t: t = t.replace("ชาต ", "ชาติ ")
    if "อนาคต" not in t: t = t.replace("อนาคด", "อนาคต")
    if "ใหม่" not in t: t = t.replace("ใหม", "ใหม่")
    if "พร้อม" not in t: t = t.replace("พรอม", "พร้อม")
    if "กรีน" not in t: t = t.replace("กรน", "กรีน").replace("กร์น", "กรีน")
    if "ไทย" not in t: t = t.replace("ทย", "ไทย")
    return t

def process_image_hybrid_v3(img_path, valid_parties, doc_type):
    result = reader.readtext(
        img_path, mag_ratio=2.0, contrast_ths=0.2, adjust_contrast=0.6
    )

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []
    current_group = []
    y_threshold = 25 if doc_type == 'party_list' else 35

    for item in lines:
        if not current_group:
            current_group.append(item)
        else:
            if abs(item['y'] - current_group[-1]['y']) < y_threshold:
                current_group.append(item)
            else:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))
                current_group = [item]

    if current_group:
        current_group.sort(key=lambda i: i['x'])
        grouped_lines.append(" ".join([i['text'] for i in current_group]))

    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        text_fixed = fix_thai_typos(text_no_num)

        best_match = process.extractOne(text_fixed, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 50:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

# ==========================================
# 🚀 ลุยรันทั้งโฟลเดอร์ 847 ไฟล์
# ==========================================

file_list = sorted(glob.glob("/content/final_data/images/*.png"))
final_submission = {row_id: 0 for row_id in df_template['id']}

print(f"🚀 เริ่มสแกนระบบ Hybrid V.3 {len(file_list)} ไฟล์ ด้วย Template ใหม่...")

for img_path in tqdm(file_list):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    doc_type = "party_list" if "party_list" in filename else "constituency"

    # 🎯 ใช้ Template ใหม่ในการดึงชื่อพรรค
    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    extracted_votes = process_image_hybrid_v3(img_path, valid_parties, doc_type)

    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

# เซฟผลลัพธ์เป็นฟอร์แมต 2 คอลัมน์พร้อมส่ง
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_final_v2.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบสมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_final_v2.csv")
print("="*40)

⏳ กำลังโหลด EasyOCR Hybrid (V.3 + อัปเดต Template V.2)...
🚀 เริ่มสแกนระบบ Hybrid V.3 846 ไฟล์ ด้วย Template ใหม่...


  0%|          | 0/846 [00:00<?, ?it/s]


✅ ปิดจ๊อบสมบูรณ์! เติมคะแนนได้ 0/10053 แถว
📂 เซฟไฟล์พร้อมส่ง: /content/submission_final_v2.csv


ส่องตาราง

In [ ]:
import easyocr
import cv2
import matplotlib.pyplot as plt

# 1. ระบุรูป Party List ที่มีปัญหา (ลองเปลี่ยนชื่อไฟล์ดูครับ)
test_image = "/content/data/images/constituency_10_10.png"

reader = easyocr.Reader(['th', 'en'], gpu=True)
result = reader.readtext(test_image)

print("--- สิ่งที่ EasyOCR มองเห็นแบบดิบๆ ---")
lines = []
for bbox, text, _ in result:
    y_center = (bbox[0][1] + bbox[2][1]) / 2
    lines.append({'text': text, 'y': y_center})

lines.sort(key=lambda x: x['y'])

# ลองพิมพ์ออกมาดู 20 บรรทัดแรก
for i, line in enumerate(lines[:20]):
    print(f"แกน Y: {line['y']:.1f} | ข้อความ: {line['text']}")

--- สิ่งที่ EasyOCR มองเห็นแบบดิบๆ ---
แกน Y: 192.5 | ข้อความ: ส.ส. ๖/๑
แกน Y: 542.0 | ข้อความ: ประกาศคณะกรรมการการเลือกตั้งประจำเขตเลือกตั้งที่
แกน Y: 547.5 | ข้อความ: ๑0
แกน Y: 623.0 | ข้อความ:  กรุงเทพมหานคร
แกน Y: 679.5 | ข้อความ: เรื่อง ผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง
แกน Y: 867.5 | ข้อความ: s และคณะกรรมการ
แกน Y: 868.5 | ข้อความ: ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร
แกน Y: 917.5 | ข้อความ: ง.
แกน Y: 931.5 | ข้อความ: เป็นวันเลือกตั้ง
แกน Y: 938.5 | ข้อความ: การเลือกตั้งได้กำหนดให้วันที่ ๘ เดือน กุมภาพันธ์ พ.ศ. ๒๕๖๙
แกน Y: 939.5 | ข้อความ: นน
แกน Y: 1024.5 | ข้อความ: บัดนี้ คณะกรรมการการเลือกตั้งประจำเขตเลือกตั้งที่
แกน Y: 1030.5 | ข้อความ:  กรุงเทพมหานคร ได้ดำเนินการรวมผล
แกน Y: 1034.0 | ข้อความ: ๑0
แกน Y: 1096.5 | ข้อความ: _เจึงขอประกาศผลการนับ
แกน Y: 1099.5 | ข้อความ: ดงนัน
แกน Y: 1104.0 | ข้อความ: -คะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้งเสร็จสิ้นเป็นที่เรียบร้อยแล้า
แกน Y: 1170.5 | ข้อความ: กรแบบแบ่งเขตเลือกตั้ง ดังนี้
แกน Y: 1

In [ ]:
import easyocr

# 1. ลองหาไฟล์ที่เป็น Party List แบบ "หน้า 2" หรือ "หน้า 3"
# (เพราะหน้า 1 มักจะเป็นหัวกระดาษ หน้า 2-3 จะเป็นตารางพรรคอัดแน่นๆ ครับ)
test_image = "/content/data/images/constituency_10_10_page2.png" # ลองเปลี่ยนไฟล์ดูครับ

reader = easyocr.Reader(['th', 'en'], gpu=True)
result = reader.readtext(test_image)

lines = []
for bbox, text, _ in result:
    y_center = (bbox[0][1] + bbox[2][1]) / 2
    lines.append({'text': text, 'y': y_center})

lines.sort(key=lambda x: x['y'])

print("--- สิ่งที่ EasyOCR มองเห็นในโซนตารางคะแนน ---")
# ขอส่องบรรทัดที่ 15 ถึง 45 (ช่วงที่เป็นรายชื่อพรรค)
for i, line in enumerate(lines[15:45]):
    print(f"แกน Y: {line['y']:.1f} | ข้อความ: {line['text']}")

--- สิ่งที่ EasyOCR มองเห็นในโซนตารางคะแนน ---
แกน Y: 764.0 | ข้อความ:  ๖
แกน Y: 836.0 | ข้อความ: ๑๙,๑๔๗ (หนึ่งหมื่นเก้าพันสี่สิบเจ็ด)
แกน Y: 842.0 | ข้อความ: เพื่อไทย
แกน Y: 858.5 | ข้อความ: โหสกุล
แกน Y: 862.0 | ข้อความ: นายภูมิพัฒน์
แกน Y: 938.5 | ข้อความ: (เก้าพันสี่ร้อยสี่สิบ)
แกน Y: 950.5 | ข้อความ: โอกาสใหม่
แกน Y: 953.0 | ข้อความ: ๙,๔๔0
แกน Y: 962.5 | ข้อความ: นางกนกนุช กลินสังข์
แกน Y: 1053.0 | ข้อความ: เจไทย
แกน Y: 1057.5 | ข้อความ: ๗,๙๒๕ (เจ็ดพันเก้าร้อยยี่สิบห้า)
แกน Y: 1057.5 | ข้อความ: ภูมิใ
แกน Y: 1064.0 | ข้อความ: นายรณกร เชียรวิชัย
แกน Y: 1077.5 | ข้อความ: ๓
แกน Y: 1158.5 | ข้อความ: ประชาธิปัตย์
แกน Y: 1159.0 | ข้อความ: ไหกพันสามร้อยเจ็ดสิบสอง)
แกน Y: 1168.5 | ข้อความ: นายชัยพร แก้ววาตะ
แกน Y: 1176.5 | ข้อความ: ๖,๓๗๒
แกน Y: 1261.0 | ข้อความ: รวมไทยสร้างชาติ
แกน Y: 1267.5 | ข้อความ: ๒,0๑๒  (สองพันสิบสอง)
แกน Y: 1270.5 | ข้อความ: นายภพิพัฒน์ สาวพัฒนะธาดา
แกน Y: 1365.5 | ข้อความ: (หนึ่งพันสี่ร้อยสามสิบเจ็ด)
แกน Y: 1374.5 | ข้อความ:  เศรษฐกิจ
แกน Y: 1377.0 | ข้อความ: นายชณ

In [ ]:
import easyocr
import glob
import os

print("🔍 กำลังค้นหาไฟล์ Party List แบบอัตโนมัติ...")
# ล็อกเป้าหาหน้า Party List หน้า 2 (ที่มีตารางพรรคแน่นๆ)
party_list_files = glob.glob("/content/data/images/party_list_*_page2.png")

if party_list_files:
    test_image = party_list_files[0]
    print(f"🎯 เจอไฟล์เป้าหมายแล้ว: {os.path.basename(test_image)}")

    reader = easyocr.Reader(['th', 'en'], gpu=True)
    result = reader.readtext(test_image, mag_ratio=2.0)

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda x: x['y'])

    print("\n--- สิ่งที่ AI มองเห็น (สังเกตแกน X ว่ามันมีคอลัมน์ซ้าย-ขวาไหม) ---")
    # ปริ้นท์ 30 บรรทัดแรกมาส่องกันครับ
    for line in lines[:30]:
        print(f"แกน Y: {line['y']:<6.1f} | แกน X: {line['x']:<6.1f} | {line['text']}")
else:
    print("❌ หาไฟล์ไม่เจอครับ ตรวจสอบ Path อีกครั้ง")

🔍 กำลังค้นหาไฟล์ Party List แบบอัตโนมัติ...
🎯 เจอไฟล์เป้าหมายแล้ว: party_list_32_6_page2.png

--- สิ่งที่ AI มองเห็น (สังเกตแกน X ว่ามันมีคอลัมน์ซ้าย-ขวาไหม) ---
แกน Y: 290.0  | แกน X: 417.5  | หมายเลข
แกน Y: 294.0  | แกน X: 858.5  | ชื่อ
แกน Y: 298.0  | แกน X: 1643.5 |  ได้คะแนน
แกน Y: 330.5  | แกน X: 417.0  | ของบัญชีรายชื่อ
แกน Y: 340.5  | แกน X: 2161.0 | หมายเหตุ
แกน Y: 357.0  | แกน X: 1643.5 | (ให้กรอกทั้งดัวเลขและตัวอักษร)
แกน Y: 363.0  | แกน X: 857.0  |  พรรคการเมือง
แกน Y: 371.0  | แกน X: 417.5  | ของพรรคการเมือง
แกน Y: 448.5  | แกน X: 1754.5 | (แปดร้อยเจ็ดสิบสอง)
แกน Y: 452.5  | แกน X: 595.5  | รวมใจไทย
แกน Y: 457.0  | แกน X: 1552.0 | ๘๗๒
แกน Y: 522.5  | แกน X: 1716.0 | (หนึ่งพันหกสิบ)
แกน Y: 528.0  | แกน X: 644.5  | รวมไทยสร้างชาติ
แกน Y: 533.5  | แกน X: 1538.0 | ด,0๖0
แกน Y: 598.0  | แกน X: 1735.5 | (หนึ่งร้อยสี่สิบเอ็ด)
แกน Y: 605.0  | แกน X: 564.0  | พลวัต
แกน Y: 610.5  | แกน X: 1565.5 | ๔๑
แกน Y: 673.5  | แกน X: 1745.0 | (ห้าร้อยสามสิบเอ็ด)
แกน Y: 679.0  | แกน X: 650.5  |

fliter *ตาราง* เอาหน้าปกกับลายเซ็นต์ออก

In [ ]:
import cv2
import numpy as np
import glob
import os
from tqdm.auto import tqdm

def has_table(img_path, min_area_ratio=0.10):
    """
    ฟังก์ชันเช็คว่าภาพนี้เป็น 'หน้าตาราง' หรือไม่
    min_area_ratio = 0.10 หมายถึง ตารางต้องใหญ่เกิน 10% ของพื้นที่กระดาษ
    """
    # 1. โหลดภาพด้วย OpenCV
    img = cv2.imread(img_path)
    if img is None:
        return False

    height, width = img.shape[:2]
    image_area = height * width

    # แปลงเป็นสีเทา
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. แปลงเป็นภาพขาวดำ (เน้นเส้นให้เป็นสีขาว พื้นเป็นสีดำ)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

    # 3. สร้าง Kernel (ไม้บรรทัด) เพื่อจับเส้นตั้งและเส้นนอน
    # กำหนดให้เส้นต้องยาวอย่างน้อย 1/40 ของขนาดภาพ ถึงจะนับว่าเป็นเส้นตาราง
    h_len = width // 40
    v_len = height // 40

    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (h_len, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, v_len))

    # 4. สกัดเฉพาะเส้นออกมา (ตัวหนังสือจะหายไปหมด)
    horizontal_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    vertical_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel, iterations=2)

    # 5. ประกอบเส้นตั้งและเส้นนอนเข้าด้วยกันเป็น Grid Mask
    table_mask = cv2.addWeighted(horizontal_lines, 0.5, vertical_lines, 0.5, 0.0)
    _, table_mask = cv2.threshold(table_mask, 50, 255, cv2.THRESH_BINARY)

    # 6. หาโครงร่าง (Contours) ของตาราง
    contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return False # ไม่มีเส้นอะไรเลย = ไม่ใช่ตาราง

    # 7. หากล่องตารางที่ใหญ่ที่สุด
    max_table_area = 0
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        max_table_area = max(max_table_area, w * h)

    # 8. ตัดสินใจ: ถ้าตารางที่ใหญ่ที่สุด กินพื้นที่เกิน 10% ของกระดาษ ถือว่าสอบผ่าน!
    if (max_table_area / image_area) > min_area_ratio:
        return True

    return False

# ==========================================
# 🚀 ลองใช้งานจริงเพื่อคัดกรองไฟล์
# ==========================================
all_files = sorted(glob.glob("/content/data/images/*.png"))
table_files = []
garbage_files = []

print(f"🔍 เริ่มสแกนคัดกรองหน้ากระดาษจากทั้งหมด {len(all_files)} ไฟล์...")

for img_path in tqdm(all_files):
    if has_table(img_path):
        table_files.append(img_path)
    else:
        garbage_files.append(img_path)

print("\n" + "="*40)
print(f"📊 สรุปผลการคัดกรอง:")
print(f"✅ เจอหน้าตาราง (นำไปใช้ต่อ): {len(table_files)} ไฟล์")
print(f"🗑️ เจอหน้าปก/ลายเซ็น (คัดทิ้ง): {len(garbage_files)} ไฟล์")
print("="*40)

# ตอนนี้สามารถเอา list `table_files` ไปใช้ในลูป OCR ต่อได้เลยครับ!

🔍 เริ่มสแกนคัดกรองหน้ากระดาษจากทั้งหมด 847 ไฟล์...


  0%|          | 0/847 [00:00<?, ?it/s]


📊 สรุปผลการคัดกรอง:
✅ เจอหน้าตาราง (นำไปใช้ต่อ): 599 ไฟล์
🗑️ เจอหน้าปก/ลายเซ็น (คัดทิ้ง): 248 ไฟล์


In [ ]:
!pip install pytesseract

In [ ]:
import cv2
import easyocr
import pandas as pd
import re
import os
import glob
from rapidfuzz import process, fuzz
from tqdm.auto import tqdm

print("⏳ กำลังเตรียมระบบ God Mode (OpenCV Cropper + EasyOCR)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

# 1. โหลด Template
df_template = pd.read_csv('/content/data/submission_template.csv')
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)
    t_clean = re.sub(r'\s*,\s*', '', t)
    nums = re.findall(r'\d+', t_clean)
    if nums:
        return int(nums[-1])
    return 0

def process_image_god_mode(img_path, valid_parties, doc_type):
    # --- 1. กรรไกรอัจฉริยะตัดตารางด้วย OpenCV ---
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

    h_len, v_len = img.shape[1]//40, img.shape[0]//40
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (h_len, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, v_len))

    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel, iterations=2)
    table_mask = cv2.addWeighted(h_lines, 0.5, v_lines, 0.5, 0.0)

    contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    max_area = 0
    table_bbox = None
    for c in contours:
        area = cv2.contourArea(c)
        if area > 50000:
            x, y, w, h = cv2.boundingRect(c)
            if area > max_area:
                max_area = area
                table_bbox = (x, y, w, h)

    if table_bbox:
        x, y, w, h = table_bbox
        pad = 20 # เผื่อขอบ 20px ให้ EasyOCR หายใจ
        cropped_img = img[max(0, y-pad) : y+h+pad, max(0, x-pad) : x+w+pad]
    else:
        cropped_img = img # ถ้าไม่มีตาราง (เผื่อหลุดมา) ก็ส่งรูปเต็ม

    # --- 2. ส่งรูปที่ตัดแล้วให้ EasyOCR อ่าน ---
    # โยน Numpy Array (cropped_img) เข้าไปตรงๆ ได้เลย!
    result = reader.readtext(
        cropped_img,
        mag_ratio=2.0,
        contrast_ths=0.2,
        adjust_contrast=0.6
    )

    # --- 3. จัดกลุ่มบรรทัด (ตาข่ายอัจฉริยะ) ---
    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []
    current_group = []

    y_threshold = 25 if doc_type == 'party_list' else 35

    for item in lines:
        if not current_group:
            current_group.append(item)
        else:
            if abs(item['y'] - current_group[-1]['y']) < y_threshold:
                current_group.append(item)
            else:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))
                current_group = [item]

    if current_group:
        current_group.sort(key=lambda i: i['x'])
        grouped_lines.append(" ".join([i['text'] for i in current_group]))

    # --- 4. สกัดคะแนน ---
    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        best_match = process.extractOne(text_no_num, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 50:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

# ==========================================
# 🚀 5. ลุยรันกับไฟล์ 599 ไฟล์ที่คัดกรองแล้ว
# ==========================================
final_submission = {row_id: 0 for row_id in df_template['id']}

# วนลูปเฉพาะใน table_files (599 ไฟล์ที่คัดไว้)
print(f"🚀 เริ่มกวาดตารางและสกัดคะแนน {len(table_files)} ไฟล์...")
for img_path in tqdm(table_files):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    doc_type = "party_list" if "party_list" in doc_id else "constituency"
    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    extracted_votes = process_image_god_mode(img_path, valid_parties, doc_type)

    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

# เซฟผลลัพธ์
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_god_mode.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบ God Mode สมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_god_mode.csv")
print("="*40)

⏳ กำลังเตรียมระบบ God Mode (OpenCV Cropper + EasyOCR)...
🚀 เริ่มกวาดตารางและสกัดคะแนน 599 ไฟล์...


  0%|          | 0/599 [00:00<?, ?it/s]


✅ ปิดจ๊อบ God Mode สมบูรณ์! เติมคะแนนได้ 8825/10053 แถว
📂 เซฟไฟล์พร้อมส่ง: /content/submission_god_mode.csv


In [ ]:
import pandas as pd

print("🔍 เริ่มกระบวนการชันสูตรข้อมูล (Error Analysis) V.2...")

# 1. โหลดข้อมูล Hybrid ของเรา (ตัวที่ดีที่สุด)
df_hybrid = pd.read_csv('/content/submission_hybrid.csv')

# 2. โหลด Template ข้อมูล
df_template = pd.read_csv('/content/data/submission_template.csv')

# 🎯 จุดที่แก้บั๊ก: สั่งเตะคอลัมน์ 'votes' ของ Template ทิ้งไปก่อน (ถ้ามี) จะได้ไม่ชนกัน!
df_template_clean = df_template.drop(columns=['votes'], errors='ignore')

# 3. รวมร่าง (Merge) ตอนนี้จะมีคอลัมน์ votes แค่อันเดียวแล้ว
df_merged = pd.merge(df_hybrid, df_template_clean, on='id', how='left')

# 4. กรองเอาเฉพาะพวกที่ได้ 0 คะแนน (ตอนนี้เรียก 'votes' ได้ตามปกติแล้ว)
df_zero = df_merged[df_merged['votes'] == 0]

# 5. แยกวิเคราะห์ระหว่าง ส.ส.เขต และ ปาร์ตี้ลิสต์
df_zero_constituency = df_zero[df_zero['id'].str.startswith('constituency')]
df_zero_partylist = df_zero[df_zero['id'].str.startswith('party_list')]

print("\n" + "="*40)
print(f"📊 สรุปยอดผู้สูญหายทั้งหมด: {len(df_zero)} แถว")
print(f"   - ส.ส.เขต: {len(df_zero_constituency)} แถว")
print(f"   - ปาร์ตี้ลิสต์: {len(df_zero_partylist)} แถว")
print("="*40)

# 6. ไฮไลท์สำคัญ: พรรคไหนในปาร์ตี้ลิสต์ที่หายไปมากที่สุด 15 อันดับแรก!
print("\n🚨 Top 15 พรรคปาร์ตี้ลิสต์ที่ AI อ่านไม่ออกมากที่สุด:")
top_missing_parties = df_zero_partylist['party_name'].value_counts().head(15)
for party, count in top_missing_parties.items():
    print(f"   ❌ {party}: หายไป {count} ครั้ง")

🔍 เริ่มกระบวนการชันสูตรข้อมูล (Error Analysis) V.2...

📊 สรุปยอดผู้สูญหายทั้งหมด: 1170 แถว
   - ส.ส.เขต: 279 แถว
   - ปาร์ตี้ลิสต์: 891 แถว

🚨 Top 15 พรรคปาร์ตี้ลิสต์ที่ AI อ่านไม่ออกมากที่สุด:
   ❌ กรีน: หายไป 53 ครั้ง
   ❌ เพื่อไทย: หายไป 43 ครั้ง
   ❌ ประชาอาสาชาติ: หายไป 35 ครั้ง
   ❌ เพื่อชาติไทย: หายไป 28 ครั้ง
   ❌ พลวัต: หายไป 27 ครั้ง
   ❌ ภูมิใจไทย: หายไป 26 ครั้ง
   ❌ อนาคตไทย: หายไป 25 ครั้ง
   ❌ เครือข่ายชาวนาแห่งประเทศไทย: หายไป 24 ครั้ง
   ❌ มิติใหม่: หายไป 24 ครั้ง
   ❌ พร้อม: หายไป 23 ครั้ง
   ❌ เพื่อบ้านเมือง: หายไป 23 ครั้ง
   ❌ ท้องที่ไทย: หายไป 23 ครั้ง
   ❌ ไทยชนะ: หายไป 23 ครั้ง
   ❌ ใหม่: หายไป 21 ครั้ง
   ❌ ไทยรวมไทย: หายไป 20 ครั้ง


In [ ]:
import pandas as pd
import easyocr
from rapidfuzz import process, fuzz
import re
import glob
import os
from tqdm.auto import tqdm

print("⏳ กำลังโหลด EasyOCR Hybrid (V.2 ระบบพจนานุกรมแก้คำผิด)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

# โหลด Template
df_template = pd.read_csv('/content/submission_template.csv')
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    """ฟังก์ชันดึงตัวเลข (ปราบภาษาอังกฤษปลอมตัว + ด.เด็ก)"""
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)

    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)
    t_clean = re.sub(r'\s*,\s*', '', t)

    nums = re.findall(r'\d+', t_clean)
    if nums:
        return int(nums[-1])
    return 0

def fix_thai_typos(text):
    """🎯 ด่านกักกันโรค: ซ่อมคำผิดยอดฮิตก่อนส่งไปเทียบชื่อ"""
    # ซ่อมพวกสระ/วรรณยุกต์หาย
    t = text.replace("เพือ", "เพื่อ").replace("ภูมใจ", "ภูมิใจ")
    t = t.replace("ชาต ", "ชาติ ").replace("ประขา", "ประชา")

    # ซ่อมพรรคชื่อสั้น และพรรคยอดฮิตที่ตกหล่น
    t = t.replace("ใหม", "ใหม่").replace("พรอม", "พร้อม")
    t = t.replace("กรน", "กรีน").replace("กร์น", "กรีน")
    t = t.replace("ทย", "ไทย")
    t = t.replace("อนาคด", "อนาคต")

    # เอาพวกสัญลักษณ์ขยะที่ชอบปนมากับชื่อพรรคออก
    t = re.sub(r'[\.\-\*\_]', '', t)
    return t

def process_image_hybrid_v2(img_path, valid_parties, doc_type):
    result = reader.readtext(
        img_path,
        mag_ratio=2.0,
        contrast_ths=0.2,
        adjust_contrast=0.6
    )

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []
    current_group = []

    y_threshold = 25 if doc_type == 'party_list' else 35

    for item in lines:
        if not current_group:
            current_group.append(item)
        else:
            if abs(item['y'] - current_group[-1]['y']) < y_threshold:
                current_group.append(item)
            else:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))
                current_group = [item]

    if current_group:
        current_group.sort(key=lambda i: i['x'])
        grouped_lines.append(" ".join([i['text'] for i in current_group]))

    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        # เรียกใช้ตัวแก้คำผิด
        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        text_fixed = fix_thai_typos(text_no_num)

        # ปรับให้ RapidFuzz ใจดีขึ้น
        best_match = process.extractOne(text_fixed, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 45:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

# ==========================================
# 🚀 ลุยรันทั้งโฟลเดอร์แบบ 100%
# ==========================================
file_list = sorted(glob.glob("/content/data/images/*.png"))
final_submission = {row_id: 0 for row_id in df_template['id']}

print(f"🚀 เริ่มสแกนระบบ Hybrid V.2 {len(file_list)} ไฟล์...")

for img_path in tqdm(file_list):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    doc_type = "party_list" if "party_list" in filename else "constituency"
    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    extracted_votes = process_image_hybrid_v2(img_path, valid_parties, doc_type)

    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

# เซฟผลลัพธ์
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_hybrid.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบ Hybrid V.2 สมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
# 🎯 แก้ชื่อไฟล์ตรงนี้ เพื่อไม่ให้โหลดผิดตัวครับ!
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_hybrid.csv")
print("="*40)

⏳ กำลังโหลด EasyOCR Hybrid (V.2 ระบบพจนานุกรมแก้คำผิด)...
🚀 เริ่มสแกนระบบ Hybrid V.2 847 ไฟล์...


  0%|          | 0/847 [00:00<?, ?it/s]


✅ ปิดจ๊อบ Hybrid V.2 สมบูรณ์! เติมคะแนนได้ 7481/10053 แถว
📂 เซฟไฟล์พร้อมส่ง: /content/submission_hybrid.csv
